# What's a Tensor? 

In [2]:
import torch

In [3]:
# A single number
scalar = torch.tensor(3.14)
print("Scalar:", scalar)

# 1D tensor = vector
vector = torch.tensor([1.0, 2.0, 3.0])
print("Vector:", vector)
print("  Shape:", vector.shape)

# A grid of numbers = matrix
matrix = torch.tensor(
    [[1.0, 2.0, 3.0],
     [4.0, 5.0, 6.0],
     [7.0, 8.0, 9.0]]
)
print("Matrix:", matrix)
print("  Shape:", matrix.shape)

Scalar: tensor(3.1400)
Vector: tensor([1., 2., 3.])
  Shape: torch.Size([3])
Matrix: tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
  Shape: torch.Size([3, 3])


In [6]:
# Understand what's grad in PyTorch
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
print("y:", y)
y.backward()
print("dy/dx:", x.grad) 

# Explanation of requires_grad=True:
explanation = """
When you create a tensor in PyTorch, 
you can specify whether it should track operations for 
automatic differentiation by setting the `requires_grad` flag. 
If `requires_grad=True`, 
PyTorch will keep track of all operations performed on that tensor, 
allowing you to compute gradients later using the `backward()` method. 
This is essential for training machine learning models, 
as it enables the optimization of parameters through gradient descent. 

In the example above, we set `requires_grad=True` for the tensor `x`, 
which allows us to compute the derivative of `y` with respect to `x` when we call `y.backward()`.
"""

y: tensor(4., grad_fn=<PowBackward0>)
dy/dx: tensor(4.)


In [15]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

model = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [17]:
# What does tokenizer do?
text = "Hello, how are you?"

print("Original text:", text)
# encode
tokens = tokenizer.encode(text)
print("Tokens:", tokens)

# decode
decoded_text = tokenizer.decode(tokens)
print("Decoded text:", decoded_text)

for token in tokens:
    print(f"Token ID: {token}, Token: {tokenizer.decode([token])}")

Original text: Hello, how are you?
Tokens: [15496, 11, 703, 389, 345, 30]
Decoded text: Hello, how are you?
Token ID: 15496, Token: Hello
Token ID: 11, Token: ,
Token ID: 703, Token:  how
Token ID: 389, Token:  are
Token ID: 345, Token:  you
Token ID: 30, Token: ?


In [25]:
# Let's get the model to work...

prompt = "The capital of France is"

model = model.to("cuda")  # Move the model to GPU for faster computation
print(next(model.parameters()).device)  # Check if the model is on GPU
inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
# Note: return_tensors="pt" means we want the output to be PyTorch tensors
# it is necessary because the model expects input in the form of tensors

# forward pass
with torch.no_grad() :
    outputs = model(inputs)
    next_token_logits = outputs.logits[0, -1, :]

# Logits? Logits are the raw, 
# unnormalized scores output by the model for each possible next token.
# To get probabilities, we can apply the softmax function to the logits
probabilities = torch.softmax(next_token_logits, dim=0)
print("Probabilities for the next token:", probabilities)


cuda:0
Probabilities for the next token: tensor([1.4029e-05, 1.4335e-05, 3.7384e-07,  ..., 1.1837e-09, 2.1117e-07,
        3.3786e-06], device='cuda:0')


In [26]:
tok10 = torch.topk(probabilities, 10)
print("Top 10 token IDs:", tok10.indices)

for i in range(10): 
    token = tok10.indices[i].item()
    prob = tok10.values[i].item()
    print(f"Token ID: {token}, Token: {tokenizer.decode([token])}, Probability: {prob:.4f}")

Top 10 token IDs: tensor([ 262,  783,  257, 4881, 6342,  287,  635,  407, 1363,  991],
       device='cuda:0')
Token ID: 262, Token:  the, Probability: 0.0846
Token ID: 783, Token:  now, Probability: 0.0479
Token ID: 257, Token:  a, Probability: 0.0462
Token ID: 4881, Token:  France, Probability: 0.0324
Token ID: 6342, Token:  Paris, Probability: 0.0322
Token ID: 287, Token:  in, Probability: 0.0266
Token ID: 635, Token:  also, Probability: 0.0264
Token ID: 407, Token:  not, Probability: 0.0238
Token ID: 1363, Token:  home, Probability: 0.0233
Token ID: 991, Token:  still, Probability: 0.0155
